# Open-Meteo - warstwa BRONZE

Cel: zamiana plików JSON z pogodą (jeden na stację mierzącą PM2.5) na jedną tabelę i zapis do Parquet.

Źródło: Open-Meteo Historical Weather API, licencja CC BY 4.0.

## 1. Ścieżki i pliki źródłowe

Wczytujemy listę plików JSON pobranych w notebooku 01

In [1]:
from pathlib import Path
import json
import runpy

import polars as pl

cwd = Path.cwd().resolve()
ROOT = cwd.parent if cwd.name == 'notebooks' else cwd
RAW_DIR = ROOT / 'data' / 'raw' / 'openmeteo' / '2024'

paths = sorted(
    path for path in RAW_DIR.glob('*.json')
    if not path.name.endswith('.metadata.json')
)

print('Plików JSON:', len(paths))
print([path.name for path in paths[:5]], '...')

Plików JSON: 96
['DsChojnowMalMOB.json', 'DsJelGorOgin.json', 'DsKlodzSzkol.json', 'DsKudowaSzkoMOB.json', 'DsWrocAlWisn.json'] ...


## 2. Jak wygląda jedna odpowiedź

Odpowiedź zawiera opis punktu (współrzędne, wysokość, jednostki) oraz część `hourly`, w której każda zmienna to lista wartości godzinowych. Czas jest w UTC.

In [2]:
payload = json.loads(paths[0].read_text(encoding='utf-8'))

print({key: value for key, value in payload.items() if key != 'hourly'})
print()

for name, values in payload['hourly'].items():
    print(f'{name:22} {len(values)} wartości, np. {values[:3]}')

{'latitude': 51.28295, 'longitude': 15.8918915, 'generationtime_ms': 204.79774475097656, 'utc_offset_seconds': 0, 'timezone': 'GMT', 'timezone_abbreviation': 'GMT', 'elevation': 152.0, 'hourly_units': {'time': 'iso8601', 'temperature_2m': '°C', 'relative_humidity_2m': '%', 'precipitation': 'mm', 'surface_pressure': 'hPa', 'wind_speed_10m': 'km/h', 'wind_direction_10m': '°'}}

time                   8784 wartości, np. ['2024-01-01T00:00', '2024-01-01T01:00', '2024-01-01T02:00']
temperature_2m         8784 wartości, np. [5.9, 6.8, 6.6]
relative_humidity_2m   8784 wartości, np. [75, 64, 64]
precipitation          8784 wartości, np. [0.1, 0.0, 0.0]
surface_pressure       8784 wartości, np. [989.2, 988.7, 988.9]
wind_speed_10m         8784 wartości, np. [17.9, 17.2, 15.1]
wind_direction_10m     8784 wartości, np. [255, 260, 245]


## 3. Konwersja wszystkich plików

Funkcja `json_to_bronze` ze `scripts/bronze_openmeteo.py` zamienia jeden plik na tabelę, w której jeden wiersz to jedna stacja i jedna godzina. Kod stacji bierzemy z nazwy pliku, czas zamieniamy na datę, a zmienne pogodowe na liczby.

In [3]:
openmeteo_module = runpy.run_path(str(ROOT / 'scripts' / 'bronze_openmeteo.py'))
weather = openmeteo_module['main']()

print(weather.schema)
print(weather.head(3))

Pliki JSON: 96
RAZEM: (843264, 12)
Zapisano: D:\project VS code\big_data\gios-bigdata\data\bronze\openmeteo_2024.parquet (7.6 MB)
Schema([('kod_stacji', String), ('data_czas_utc', Datetime(time_unit='us', time_zone=None)), ('temperature_2m', Float64), ('relative_humidity_2m', Float64), ('precipitation', Float64), ('surface_pressure', Float64), ('wind_speed_10m', Float64), ('wind_direction_10m', Float64), ('siatka_szerokosc', Float64), ('siatka_dlugosc', Float64), ('wysokosc_m', Float64), ('plik_zrodlowy', String)])
shape: (3, 12)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ kod_stacj ┆ data_czas ┆ temperatu ┆ relative_ ┆ … ┆ siatka_sz ┆ siatka_dl ┆ wysokosc_ ┆ plik_zro │
│ i         ┆ _utc      ┆ re_2m     ┆ humidity_ ┆   ┆ erokosc   ┆ ugosc     ┆ m         ┆ dlowy    │
│ ---       ┆ ---       ┆ ---       ┆ 2m        ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---      │
│ str       ┆ datetime[ ┆ f64       ┆ ---       ┆   ┆ f64   

## 4. Kontrola

Oczekujemy 96 stacji x 8784 godziny = 843 264 wierszy, bez duplikatów i braków.

In [4]:
print('wiersze:', weather.height)
print('stacje:', weather['kod_stacji'].n_unique())
print('zakres czasu:', weather['data_czas_utc'].min(), '-', weather['data_czas_utc'].max())
print('duplikaty (stacja, godzina):', weather.select('kod_stacji', 'data_czas_utc').is_duplicated().sum())
print()
print(weather.null_count())
print(weather.select(openmeteo_module['WEATHER_COLUMNS']).describe())

wiersze: 843264
stacje: 96
zakres czasu: 2024-01-01 00:00:00 - 2024-12-31 23:00:00
duplikaty (stacja, godzina): 0

shape: (1, 12)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ kod_stacj ┆ data_czas ┆ temperatu ┆ relative_ ┆ … ┆ siatka_sz ┆ siatka_dl ┆ wysokosc_ ┆ plik_zro │
│ i         ┆ _utc      ┆ re_2m     ┆ humidity_ ┆   ┆ erokosc   ┆ ugosc     ┆ m         ┆ dlowy    │
│ ---       ┆ ---       ┆ ---       ┆ 2m        ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---      │
│ u32       ┆ u32       ┆ u32       ┆ ---       ┆   ┆ u32       ┆ u32       ┆ u32       ┆ u32      │
│           ┆           ┆           ┆ u32       ┆   ┆           ┆           ┆           ┆          │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 0         ┆ 0         ┆ 0         ┆ 0         ┆ … ┆ 0         ┆ 0         ┆ 0         ┆ 0        │
└───────────┴───────────┴───────────┴───────────┴───┴─────────